# Classification: Cane Corso Growth Status

This notebook applies the second course topic: **Classification**.

The goal is to use the processed real public dog growth sample and build classification models step by step.

The classification task will not provide veterinary diagnosis. It will only classify records into educational growth-status categories for machine learning practice.


## Course Topic Coverage

This notebook will follow the Classification lecture step by step:

1. Classification problem statement and motivation
2. Logistic Regression
3. Binary classification
4. Data preparation: encoding, scaling, train/test split, class balance
5. Evaluation: confusion matrix, accuracy, precision, recall, F1
6. ROC curve and AUC
7. Decision Trees
8. Ensemble models
9. Support Vector Machines
10. Final model comparison and interpretation


## Problem Statement

In the previous notebook, the project used regression to predict a numerical value: dog weight.

In this notebook, the task changes from predicting a number to predicting a class.

The planned classification target is `growth_status`:

- `normal_growth`
- `needs_attention`

This target will be created from body condition score information in the processed real dataset.


## Dataset

This notebook uses the processed real public dog growth sample:

`data/processed/dog_growth_public_sample.csv`

The original raw dataset is not committed to GitHub. Only the smaller processed sample is used in this notebook.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)


In [2]:
data_path = '../data/processed/dog_growth_public_sample.csv'
df = pd.read_csv(data_path)
df.head()


,breed_id,pet_id,gender,visit_age_years,visit_age_months,weight_kg,bcs_recorded,bcs_predicted,preventive_care_visit,healthy_pet_diagnosis,average_adult_breed_weight_kg,source_type
0,11,952168,F,0.298,3.58,16.919,NaN,NaN,Y,Y,35.91,real_public_processed_sample
1,11,952169,M,0.169,2.03,7.938,Thin,NaN,N,N,35.91,real_public_processed_sample
2,11,952169,M,0.355,4.26,12.610,NaN,NaN,Y,N,35.91,real_public_processed_sample
3,11,952170,FS,0.355,4.26,16.964,NaN,NaN,N,N,35.91,real_public_processed_sample
4,11,952170,FS,0.421,5.05,21.364,NaN,NaN,Y,Y,35.91,real_public_processed_sample


## Initial Dataset Check

Before creating the classification target, I first inspect the dataset shape, columns, and basic values.


In [3]:
df.shape


(10000, 12)

In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   breed_id                       10000 non-null  int64  
 1   pet_id                         10000 non-null  int64  
 2   gender                         10000 non-null  str    
 3   visit_age_years                10000 non-null  float64
 4   visit_age_months               10000 non-null  float64
 5   weight_kg                      10000 non-null  float64
 6   bcs_recorded                   324 non-null    str    
 7   bcs_predicted                  421 non-null    str    
 8   preventive_care_visit          10000 non-null  str    
 9   healthy_pet_diagnosis          10000 non-null  str    
 10  average_adult_breed_weight_kg  10000 non-null  float64
 11  source_type                    10000 non-null  str    
dtypes: float64(4), int64(2), str(6)
memory usage: 937.6 KB


In [5]:
df[['bcs_recorded', 'bcs_predicted', 'source_type']].head(10)


,bcs_recorded,bcs_predicted,source_type
0,NaN,NaN,real_public_processed_sample
1,Thin,NaN,real_public_processed_sample
2,NaN,NaN,real_public_processed_sample
3,NaN,NaN,real_public_processed_sample
4,NaN,NaN,real_public_processed_sample
5,NaN,NaN,real_public_processed_sample
6,NaN,NaN,real_public_processed_sample
7,NaN,NaN,real_public_processed_sample
8,NaN,NaN,real_public_processed_sample
9,NaN,NaN,real_public_processed_sample


## Create Classification Target

For this classification task, I create a new target column called `growth_status`.

The target is based on body condition score information:

- `Normal` becomes `normal_growth`
- `Thin` or `Heavy` becomes `needs_attention`

This is not a veterinary diagnosis. It is only an educational classification label used to practice machine learning methods.

In [6]:
df["bcs_source"] = df["bcs_recorded"]

df.loc[
    (df["bcs_source"].isna()) | (df["bcs_source"] == "None"),
    "bcs_source"
] = df["bcs_predicted"]

df["growth_status"] = df["bcs_source"].map({
    "Normal": "normal_growth",
    "Thin": "needs_attention",
    "Heavy": "needs_attention"
})

classification_df = df.dropna(subset=["growth_status"]).copy()

classification_df[["bcs_recorded", "bcs_predicted", "bcs_source", "growth_status"]].head(10)

,bcs_recorded,bcs_predicted,bcs_source,growth_status
1,Thin,NaN,Thin,needs_attention
24,NaN,Thin,Thin,needs_attention
43,Heavy,NaN,Heavy,needs_attention
84,NaN,Thin,Thin,needs_attention
126,Thin,NaN,Thin,needs_attention
127,Thin,NaN,Thin,needs_attention
128,Thin,NaN,Thin,needs_attention
130,Thin,NaN,Thin,needs_attention
131,Thin,NaN,Thin,needs_attention
132,Thin,NaN,Thin,needs_attention


In [7]:
classification_df["growth_status"].value_counts()

growth_status
needs_attention    579
normal_growth      166
Name: count, dtype: int64

In [8]:
classification_df["growth_status"].value_counts(normalize=True).round(3)

growth_status
needs_attention    0.777
normal_growth      0.223
Name: proportion, dtype: float64

In [9]:
classification_df["growth_status_binary"] = classification_df["growth_status"].map({
    "normal_growth": 0,
    "needs_attention": 1
})

classification_df[["growth_status", "growth_status_binary"]].head()

,growth_status,growth_status_binary
1,needs_attention,1
24,needs_attention,1
43,needs_attention,1
84,needs_attention,1
126,needs_attention,1


### Target Interpretation

The `growth_status_binary` column is the numeric target for binary classification.

- `0` means `normal_growth`
- `1` means `needs_attention`

This target allows the project to apply Logistic Regression and other classification models from the course.